## microk8s

MicroK8s ist eine leichtgewichtige, produktionsbereite Kubernetes-Distribution, die von Canonical entwickelt wurde. Sie eignet sich ideal für Entwickler, die eine schnelle und einfache Möglichkeit suchen, Kubernetes in einer lokalen oder kleinen Umgebungen zu testen oder zu betreiben.


### Kubernetes-Labels für Master/Control-Plane setzen (nur microk8s)

Einige cncf.io Projekte erwarten, dass die Control-Plane als `master` bzw. `control-plane` gelabelt ist.

Ansonsten kann es passieren, dass die Pods nicht starten.


In [ ]:
%%bash
kubectl label nodes $(kubectl get nodes -o custom-columns=NAME:.metadata.name | awk 'NR==2') node-role.kubernetes.io/master=
kubectl label nodes $(kubectl get nodes -o custom-columns=NAME:.metadata.name | awk 'NR==2') node-role.kubernetes.io/control-plane=


## Dashboard

In [ ]:
%%bash
echo "https://$(cat ~/work/server-ip)":30443  

## Worker Nodes joinen

In [ ]:
import os
os.environ['ControlPlane']='ip-172-31-28-204.ec2.internal'
os.environ['Worker1']='ip-172-31-22-186.ec2.internal'
os.environ['Worker2']='ip-172-31-29-92.ec2.internal'

In [ ]:
%%bash
JOIN=$(sudo microk8s add-node --token-ttl 3600 | head -2 | tail -1)
ssh $Worker1 -- sudo $JOIN

In [ ]:
%%bash
JOIN=$(sudo microk8s add-node --token-ttl 3600 | head -2 | tail -1)
ssh $Worker2 -- sudo $JOIN

In [ ]:
%%bash
kubectl get nodes

### Persistenz

Auf der ersten Node (**controlplane**) wurden bei der Installation folgendes eingerichtet:
* **Storage Class** `local-storage` 
* **Persistent Volume** `rwm-volume` eingerichtet, welches die Daten in `/data` speichert
* Eine **NFS Freigabe** von `/data`
* Ein **Persistent Volume Claim** `data-claim` welcher auf `rwm-volume` zeigt und gleichzeitigen Zugriff von Unterschiedlichen Pods zulässt (Read-Write-Many).

Damit **worker1** und **worker2** ebenfalls Zugriff auf `/data` haben werden dessen `/data`-Verzeichnisse mit **controlplane** gemountet.

In [ ]:
%%bash
ssh $Worker1 "sudo mkdir -p /data; sudo mount -t nfs $ControlPlane:/data /data"

In [ ]:
%%bash
ssh $Worker2 "sudo mkdir -p /data; sudo mount -t nfs $ControlPlane:/data /data"